In [10]:
import pandas as pd
from model import StandingOvationModel

In [11]:
def get_agents(model):
    agents = []
    for x in range(model.grid.width):
        for y in range(model.grid.height):
            cell = model.grid.get_cell_list_contents([(x, y)])
            agents.extend(cell)
    return agents

In [12]:
def run_once(neighborhood, order, seed, max_steps=50):

    model = StandingOvationModel(
        neighborhood_type=neighborhood,
        order=order,
        seed=seed
    )

    history = []

    for step in range(max_steps):
        model.step()

        agents = get_agents(model)
        standing = sum(a.standing for a in agents)
        history.append(standing)

        if step > 0 and history[-1] == history[-2]:
            break

    agents = get_agents(model)
    N = len(agents)

    NI = step + 1

    initial_majority = history[0] > N / 2
    final_majority = history[-1] > N / 2

    IE = 1 if initial_majority == final_majority else 0

    final_state = final_majority
    SM = sum(a.standing != final_state for a in agents) / N

    return NI, SM, IE

In [13]:
def experiment(neighborhood, order, runs=100):

    results = []

    for seed in range(runs):
        results.append(run_once(neighborhood, order, seed))

    df = pd.DataFrame(results, columns=["NI", "SM", "IE"])

    return {
        "NI": df["NI"].mean(),
        "SM": df["SM"].mean() * 100,
        "IE": df["IE"].mean() * 100
    }

In [14]:
configs = [
    ("five", "Random"),
    ("five", "Synchronous"),
    ("five", "Incentive"),
    ("cone", "Random"),
    ("cone", "Synchronous"),
    ("cone", "Incentive"),
]

In [15]:
table = []

for neighborhood, order in configs:
    stats = experiment(neighborhood, order)

    table.append({
        "Neighborhood": neighborhood,
        "Update": order,
        "NI": stats["NI"],
        "SM": stats["SM"],
        "IE": stats["IE"]
    })

results_df = pd.DataFrame(table)
results_df

,Neighborhood,Update,NI,SM,IE
0,five,Random,19.58,20.9425,64.0
1,five,Synchronous,20.78,26.1825,72.0
2,five,Incentive,16.10,6.2750,65.0
3,cone,Random,19.60,16.3750,62.0
4,cone,Synchronous,21.97,18.7075,61.0
5,cone,Incentive,11.28,3.1275,67.0
